# Inference Cost vs. Defence Performance

Aggregates all experiment results and plots **inference cost** (forward passes per image)
against **mean post-defence accuracy** for both multilingual and unilingual setups.

## Inference cost table
| Method        | Forward passes / image |
|---------------|------------------------|
| no_defense    | 2                      |
| cam_2mod      | 6                      |
| cam_4mod      | 10                     |
| grid_1patch   | 32                     |
| grid_2patch   | 62                     |

In [1]:
import json, os, glob
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

HERE = os.path.dirname(os.path.abspath('cost_vs_performance.ipynb')) if '__file__' not in dir() else os.path.dirname(os.path.abspath(__file__))

# Collect all result JSONs
patterns = [
    os.path.join('multilingual', 'results', '**', '*.json'),
    os.path.join('unilingual',   'results', '**', '*.json'),
]
all_results = []
for pat in patterns:
    for path in glob.glob(pat, recursive=True):
        with open(path, encoding='utf-8') as f:
            d = json.load(f)
        all_results.append(d)
        print(f'Loaded {path}: method={d.get("method")}, setup={d.get("setup")}')
print(f'Total result files: {len(all_results)}')

Loaded multilingual\results\attack\confusion_results.json: method=no_defense, setup=multilingual
Loaded multilingual\results\cam_2mod\confusion_results_cam_defense.json: method=cam_2mod, setup=multilingual
Loaded multilingual\results\cam_4mod\confusion_results_cam_defense.json: method=cam_4mod, setup=multilingual
Loaded multilingual\results\grid_1patch\confusion_results_grid.json: method=grid_1patch, setup=multilingual
Loaded multilingual\results\grid_2patch\confusion_results_grid.json: method=grid_2patch, setup=multilingual
Loaded unilingual\results\attack\confusion_results.json: method=no_defense, setup=unilingual
Loaded unilingual\results\cam_2mod\confusion_results_cam_defense.json: method=cam_2mod, setup=unilingual
Loaded unilingual\results\grid_1patch\confusion_results_grid.json: method=grid_1patch, setup=unilingual
Loaded unilingual\results\grid_2patch\confusion_results_grid.json: method=grid_2patch, setup=unilingual
Total result files: 9


In [2]:
# Inference cost table (forward passes per image)
COST_TABLE = {
    'no_defense':   2,
    'cam_2mod':     6,
    'cam_4mod':    10,
    'grid_1patch': 32,
    'grid_2patch': 62,
}

METHOD_ORDER = ['no_defense', 'cam_2mod', 'cam_4mod', 'grid_1patch', 'grid_2patch']
SETUP_COLORS = {'multilingual': 'C0', 'unilingual': 'C1'}
METHOD_MARKERS = {
    'no_defense':   'x',
    'cam_2mod':     'o',
    'cam_4mod':     's',
    'grid_1patch':  '^',
    'grid_2patch':  'D',
}

def get_perf(d):
    """Mean post-defence accuracy across EN and ZH models."""
    if d.get('method') == 'no_defense':
        acc = d.get('attacked_acc', d.get('defense_acc', {}))
    else:
        acc = d.get('defense', d.get('defense_acc', {}))
        if isinstance(acc, dict) and 'en' not in acc:
            # cam defence stores nested dict under 'defense'
            acc_vals = [v['acc'] for v in acc.values() if isinstance(v, dict) and 'acc' in v]
            return float(np.mean(acc_vals)) * 100 if acc_vals else None
    if isinstance(acc, dict):
        vals = []
        for v in acc.values():
            if isinstance(v, (int, float)):
                vals.append(v)
            elif isinstance(v, dict) and 'acc' in v:
                vals.append(v['acc'])
        if vals: return float(np.mean(vals)) * 100
    return None

fig, ax = plt.subplots(figsize=(8, 5))

for setup, color in SETUP_COLORS.items():
    xs, ys, labels = [], [], []
    for d in all_results:
        if d.get('setup') != setup: continue
        method = d.get('method', 'unknown')
        cost   = COST_TABLE.get(method)
        perf   = get_perf(d)
        if cost is None or perf is None: continue
        xs.append(cost); ys.append(perf); labels.append(method)
    if not xs: continue
    # Sort by cost
    order = sorted(range(len(xs)), key=lambda i: xs[i])
    xs = [xs[i] for i in order]; ys = [ys[i] for i in order]; labels = [labels[i] for i in order]
    ax.plot(xs, ys, '-', color=color, alpha=0.5, linewidth=1.2)
    for x, y, lbl in zip(xs, ys, labels):
        marker = METHOD_MARKERS.get(lbl, 'o')
        ax.scatter(x, y, color=color, marker=marker, s=80, zorder=5,
                   label=f'{setup} / {lbl}')
        ax.annotate(lbl.replace('_', ' '), (x, y), textcoords='offset points',
                    xytext=(6, 4), fontsize=7.5)

# Baseline (no-defense) reference line
baseline_results = [d for d in all_results if d.get('method') == 'no_defense']
if baseline_results:
    baseline_perfs = [p for d in baseline_results for p in [get_perf(d)] if p is not None]
    if baseline_perfs:
        ax.axhline(float(np.mean(baseline_perfs)), color='grey', linestyle=':', alpha=0.7, label='no-defense baseline')

ax.set_xlabel('Inference cost (forward passes per image)', fontsize=11)
ax.set_ylabel('Mean post-defence accuracy (%)', fontsize=11)
ax.set_title('Inference Cost vs. Defence Performance\n(multilingual vs unilingual typographic attack)', fontsize=12)
ax.grid(True, alpha=0.3)

# Legend: one entry per (setup, method) combination, no duplicates
handles, lbls = ax.get_legend_handles_labels()
seen = set(); unique = [(h, l) for h, l in zip(handles, lbls) if l not in seen and not seen.add(l)]
ax.legend([h for h, _ in unique], [l for _, l in unique], fontsize=8, loc='lower right', ncol=2)

plt.tight_layout()
plt.savefig('cost_vs_performance.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved -> cost_vs_performance.png')

Saved -> cost_vs_performance.png
